In [227]:
import re
import unicodedata
from pathlib import Path
from typing import List, Optional
import pandas as pd

In [228]:

def normalize_text(text: str) -> str:
    if text is None:
        return ""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u00a0", " ")
    return text


def clean_extracted_block(text: str) -> str:
    text = normalize_text(text)

    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'\b\d+/\d+\b', ' ', text)
    text = re.sub(r'\b\d{1,2}/\d{1,2}/\d{2,4},?\s+\d{1,2}:\d{2}\s*(?:AM|PM|am|pm)?\b', ' ', text)
    # Keep multiple spaces because some transcripts use them as speaker/content delimiters.
    text = re.sub(r'\t+', ' ', text)
    text = re.sub(r' *\n *', '\n', text)

    cleaned_lines = []
    for line in text.split("\n"):
        s = line.strip()
        if not s:
            cleaned_lines.append("")
            continue

        if re.fullmatch(
            r'(Skip to content|Search|Share|Twitter|LinkedIn|Facebook|Email|Copy Link|Related Stories|Newsletter|Subscribe|menu|search|Site Index)',
            s,
            flags=re.I,
        ):
            continue

        cleaned_lines.append(s)

    text = "\n".join(cleaned_lines)
    text = re.sub(r'\n{3,}', '\n\n', text).strip()
    return text


def build_speaker_pattern(artist_labels: List[str]) -> re.Pattern:
    escaped = sorted([re.escape(x) for x in artist_labels], key=len, reverse=True)
    label_alt = "|".join(escaped)
    return re.compile(
        rf'^\s*(?:A\s*)?(?P<label>{label_alt})\s*(?:\:|—|-)\s*(?P<content>.*)$',
        flags=re.I
    )


def build_label_only_pattern(artist_labels: List[str]) -> re.Pattern:
    escaped = sorted([re.escape(x) for x in artist_labels], key=len, reverse=True)
    label_alt = "|".join(escaped)
    return re.compile(
        rf'^\s*(?:A\s*)?(?P<label>{label_alt})\s*$',
        flags=re.I
    )


def build_speaker_spaced_pattern(artist_labels: List[str]) -> re.Pattern:
    escaped = sorted([re.escape(x) for x in artist_labels], key=len, reverse=True)
    label_alt = "|".join(escaped)
    return re.compile(
        rf'^\s*(?:A\s*)?(?P<label>{label_alt})\s{{2,}}(?P<content>\S.*)$',
        flags=re.I
    )


def extract_from_labeled_interview(text: str, artist_labels: List[str]) -> List[str]:
    text = clean_extracted_block(text)
    lines = text.split("\n")

    artist_re = build_speaker_pattern(artist_labels)
    artist_label_only_re = build_label_only_pattern(artist_labels)
    artist_spaced_re = build_speaker_spaced_pattern(artist_labels)
    any_speaker_re = re.compile(
        r'^\s*(?:Q\s*)?(?:A\s*)?[A-Z][A-Za-z ./&\'\-]{0,60}\s*(?:\:|—|-)\s*',
        flags=re.I,
    )
    any_label_only_re = re.compile(
        r'^\s*(?:Q\s*)?(?:A\s*)?[A-Z][A-Z/&\'\-]*(?:\s+[A-Z][A-Z/&\'\-]*){0,4}\s*$',
        flags=re.I,
    )
    any_spaced_re = re.compile(
        r'^\s*(?:Q\s*)?(?:A\s*)?[A-Z][A-Z/&\'\-]*(?:\s+[A-Z][A-Z/&\'\-]*){0,4}\s{2,}\S.*$',
        flags=re.I,
    )

    results = []
    current_speaker = None
    buffer = []
    PARA = "__PARA_BREAK__"

    def flush_artist_buffer() -> None:
        nonlocal buffer
        if not buffer:
            return
        parts = []
        current = []
        for tok in buffer:
            if tok == PARA:
                if current:
                    parts.append(" ".join(current).strip())
                    current = []
            else:
                current.append(tok)
        if current:
            parts.append(" ".join(current).strip())
        text_out = "\n\n".join(p for p in parts if p).strip()
        if text_out:
            results.append(text_out)
        buffer = []

    for line in lines:
        artist_match = artist_re.match(line)
        artist_label_only_match = artist_label_only_re.match(line)
        artist_spaced_match = artist_spaced_re.match(line)

        if artist_match:
            if current_speaker == "artist" and buffer:
                flush_artist_buffer()

            current_speaker = "artist"
            content = artist_match.group("content").strip()
            buffer = [content] if content else []
            continue

        elif artist_label_only_match:
            if current_speaker == "artist" and buffer:
                flush_artist_buffer()

            current_speaker = "artist"
            buffer = []
            continue

        elif artist_spaced_match:
            if current_speaker == "artist" and buffer:
                flush_artist_buffer()

            current_speaker = "artist"
            content = artist_spaced_match.group("content").strip()
            buffer = [content] if content else []
            continue

        elif any_speaker_re.match(line) or any_label_only_re.match(line) or any_spaced_re.match(line):
            if current_speaker == "artist" and buffer:
                flush_artist_buffer()
            current_speaker = "other"
            buffer = []
            continue

        else:
            if current_speaker == "artist":
                s = line.strip()
                if s:
                    buffer.append(s)
                else:
                    # Paragraph break inside the same speaker turn; preserve boundary.
                    if buffer and buffer[-1] != PARA:
                        buffer.append(PARA)
                    continue

    if current_speaker == "artist" and buffer:
        flush_artist_buffer()

    return [r for r in results if r]


def looks_like_title(quote: str, right_context: str = "") -> bool:
    words = quote.strip().split()
    if not words or len(words) > 6:
        return False

    alpha_words = [w.strip(".,:;!?()[]") for w in words if re.search(r"[A-Za-z]", w)]
    if not alpha_words:
        return False

    mostly_title_case = all(
        (w[:1].isupper() if w[:1].isalpha() else True)
        for w in alpha_words
    )

    if mostly_title_case and not re.search(r"[.?!]", quote):
        if re.match(r'^\s*(pairs|is|was|becomes|,)', right_context, flags=re.I):
            return True

    return False


def extract_quotes_article(
    text: str,
    artist_name: str,
    artist_aliases: Optional[List[str]] = None,
) -> List[str]:
    text = clean_extracted_block(text)

    normalized = (
        text.replace("“", '"')
            .replace("”", '"')
            .replace("‘", "'")
            .replace("’", "'")
    )

    if artist_aliases is None:
        artist_aliases = []

    aliases = list(dict.fromkeys([
        artist_name,
        artist_name.split()[-1],
        *artist_aliases
    ]))
    name_pat = "|".join(re.escape(a) for a in aliases if a)

    quote_re = re.compile(r'"([^"]+)"')
    matches = list(quote_re.finditer(normalized))

    attribution_verbs = (
        r'says|said|told me|told us|explains|explained|adds|added|'
        r'notes|noted|calls|called|describes|described|responds|responded|'
        r'laughs|laughed|asks|asked'
    )

    candidates = []

    for m in matches:
        quote = m.group(1).strip()
        start, end = m.span()

        left = normalized[max(0, start - 220):start]
        right = normalized[end:min(len(normalized), end + 220)]

        attributed = any([
            re.search(rf'({name_pat})\s+(?:{attribution_verbs})', left[-140:] + right[:140], flags=re.I),
            re.search(rf'(?:{attribution_verbs})\s+({name_pat})', left[-140:] + right[:140], flags=re.I),
            re.search(rf'\b(?:she|he|they)\s+(?:{attribution_verbs})\b', left[-140:] + right[:140], flags=re.I),
        ])

        para_start = normalized.rfind("\n\n", 0, start)
        para_start = 0 if para_start == -1 else para_start
        para = normalized[para_start:start]

        if not attributed:
            if re.search(rf'({name_pat})', para, flags=re.I) and re.search(
                rf'\b(?:she|he|they)\s+(?:{attribution_verbs})\b',
                para,
                flags=re.I
            ):
                attributed = True

        if attributed and not looks_like_title(quote, right):
            candidates.append((start, end, quote))

    merged = []
    i = 0
    while i < len(candidates):
        s, e, q = candidates[i]
        parts = [q]
        j = i + 1

        while j < len(candidates):
            s2, e2, q2 = candidates[j]
            bridge = normalized[e:s2]

            if len(bridge) < 90 and re.fullmatch(
                r'[\s,;:\-\u2014\.]*(?:'
                r'she says,?|he says,?|they say,?|'
                r'she said,?|he said,?|they said,?|'
                rf'{name_pat}\s+says,?|{name_pat}\s+said,?|'
                r'she laughs,?|she chuckles,?'
                r')?[\s,;:\-\u2014\.]*',
                bridge,
                flags=re.I,
            ):
                parts.append(q2)
                e = e2
                j += 1
            else:
                break

        merged.append((s, " ".join(parts).strip()))
        i = j

    integrated = []
    for pat in [
        rf'({name_pat})\s+(?:describes|described|calls|called)\s+[^.:\n]*?\bas\s+"([^"]+)"',
        rf'({name_pat})\s+(?:describes|described)\s+[^.:\n]*?\bmore about\s+"([^"]+)"',
    ]:
        for m in re.finditer(pat, normalized, flags=re.I):
            integrated.append((m.start(2), m.group(2).strip()))

    all_quotes = sorted(merged + integrated, key=lambda x: x[0])

    output = []
    seen = set()
    for _, q in all_quotes:
        q = q.strip(" \n\t-–—")
        if q and q not in seen:
            seen.add(q)
            output.append(q)

    return output


def parse_artist_words(
    text: str,
    artist_name: str,
    artist_labels: Optional[List[str]] = None,
    artist_aliases: Optional[List[str]] = None,
    mode: str = "auto",
) -> List[str]:
    text = clean_extracted_block(text)

    if artist_labels is None:
        parts = artist_name.split()
        initials = "".join(p[0] for p in parts if p)
        artist_labels = list(dict.fromkeys([
            artist_name,
            artist_name.upper(),
            parts[-1],
            parts[-1].upper(),
            initials,
            initials.upper(),
        ]))
    else:
        parts = artist_name.split()
        initials = "".join(p[0] for p in parts if p)
        extras = [artist_name, artist_name.upper()]
        if parts:
            extras.extend([parts[-1], parts[-1].upper()])
        if initials:
            extras.extend([initials, initials.upper()])
        artist_labels = list(dict.fromkeys([*artist_labels, *extras]))

    labeled_hits = 0
    for lbl in artist_labels:
        if re.search(rf'^\s*(?:A\s*)?{re.escape(lbl)}\s*(?:\:|—|-)', text, flags=re.I | re.M) or re.search(rf'^\s*(?:A\s*)?{re.escape(lbl)}\s*$', text, flags=re.I | re.M) or re.search(rf'^\s*(?:A\s*)?{re.escape(lbl)}\s{{2,}}\S', text, flags=re.I | re.M):
            labeled_hits += 1

    if mode == "interview":
        return extract_from_labeled_interview(text, artist_labels)

    if mode == "auto" and labeled_hits > 0:
        return extract_from_labeled_interview(text, artist_labels)

    if mode in ("auto", "article"):
        return extract_quotes_article(
            text=text,
            artist_name=artist_name,
            artist_aliases=artist_aliases,
        )

    return []


def remove_outer_quotes(blocks: List[str]) -> List[str]:
    cleaned = []
    for b in blocks:
        s = b.strip()
        s = re.sub(r'^[\"\']+', '', s)
        s = re.sub(r'[\"\']+$', '', s)
        cleaned.append(s.strip())
    return cleaned


def dedupe_blocks(blocks: List[str]) -> List[str]:
    seen = set()
    out = []
    for b in blocks:
        key = re.sub(r'\s+', ' ', b.strip().lower())
        if key and key not in seen:
            seen.add(key)
            out.append(b.strip())
    return out


def to_single_text(blocks: List[str], sep: str = "\n\n") -> str:
    return sep.join([b.strip() for b in blocks if b.strip()])

In [ ]:
INPUT_FILE = Path("input.txt")

artist_name = "Hung Liu"
artist_labels = ["HL", "Hung", "Liu"]
artist_aliases = ["Liu"]

mode = "interview"

raw_text = INPUT_FILE.read_text(encoding="utf-8")

In [236]:
artist_words = parse_artist_words(
    raw_text,
    artist_name=artist_name,
    artist_labels=artist_labels,
    artist_aliases=artist_aliases,
    mode=mode,
)

artist_words = dedupe_blocks(artist_words)
artist_words = remove_outer_quotes(artist_words)

In [237]:
for block in artist_words:
    print(block)

I was there for about three weeks. Beijing was where I stayed most of the time. My mom lives there. One thing we did which was unexpectedly fascinating was a family graveyard sweeping ceremony. We went back to my grandparent's hometown. She left when she was five and never came back. In Chinese tradition it is very important to have some kind of ceremony and actually to sweep your ancestor’s tomb every year to show respect. We talked about it, but never did it because of various inconveniences. My grandfather died in 62. My grandmother died in 79. They were finally buried together in the family graveyard in the 80’s.
Exactly. That's grandma. In the early sixties cremation was still a new idea. But my grandfather said, "That's okay." So after we cremated my grandmother we decided to send their ashes back to her homeland in Manchuria, to Xiao Huang Jin Cun—meaning little golden village, in northeast China. My mom and I have had the dream of going there to pay homage for years and years, 

In [64]:
print(artist_words)

['I want them to be analogous to what you experience when you’re  walking around at night in the city. Often you might be alone, but even if you’re with people you’re isolated from everyone just because it’s dark. And there’s just this kind of closeness. You look up at the sky, but it’s never that vast, forever sky.', 'Yes. Even if there’s no cloud cover, there’s still reflective light. Otherwise it would be black and we would see stars.', 'If I were trying to be realistic, if I were trying to make the Alex Katz versions of these paintings, there might be one star, or maybe Venus and a star, because when the sky is clear and there’s no moon, you do occasionally see a couple of stars, even in the city. But you can never be sure if they’re satellites or planes.', 'I never thought of it exactly that way but that’s perfect. [laughs]', 'Maybe the winter of 2009. I was painting only the field; there weren’t these borders. I’d have Glenn [Ligon] and Lisa [Segal, the artist and Kim’s wife] int